In [5]:
from joblib import Memory
from dask.distributed import Client, LocalCluster, wait, as_completed
import time
from tqdm import tqdm_notebook as tqdm

### Parallel Helper

In [1]:
# We wrap the clients in a class to prevent
# double init
class MyClient:
    mp = None
    modules = None 
    started = False

    # Stats
    p_workers = None 
    p_cores   = None 
    
    @staticmethod
    def restart():
        MyClient.mp.restart()
    
    @staticmethod
    def init(processes=True, **kwargs):
        if MyClient.started is False:
            # Start process cluster
            print("Starting cluster...")
            MyClient.mp = Client(scheduler_port=0, dashboard_address=None, processes=processes, **kwargs)
            MyClient.p_workers = len(MyClient.mp.ncores())
            MyClient.p_cores = list(MyClient.mp.ncores().values())[0]
            print("Cluster started w/ {} workers ({} cores each)".format(MyClient.p_workers, MyClient.p_cores))
            # Toggle             
            MyClient.started = True
        else:
            print("Cluster already started")

def init_parallel(module_names=None, **kwargs):
    MyClient.init(**kwargs)
    if module_names is not None:
        for name in module_names:
            MyClient.mp.upload_file(name)
    print(
        """
        Parallel Plugin Loaded. You can now decorate functions with @profile(profile_array) 
        and @parallel(map=True, background=False). MyClient and get_results(futures)
        have also been loaded into your namespace.
        """)

    
def parallel(map=False, background=True):
    def concurrent_decorator(func):
        def wrapper_concurrent(*args, **kwargs):
            client = MyClient.mp
            if map:
                res = client.map(func, *args, **kwargs)
            else:
                res = client.submit(func, *args, **kwargs)
            if background:
                return res
            else:
                # Caching bug here               
                res = [r for f, r in tqdm(as_completed(res, with_results=True), total=len(res))]
                return res 
        return wrapper_concurrent
    return concurrent_decorator

def get_results(futures):
    return [f.result() for f in as_completed(futures)]